# US Monthly Candy Production Time Series Analysis and Forecasting

**Course / Assignment:** STA 6110 / Time Series Group Assignment
**Student / Team:** Gabriel Kinyanjui Kariuki & Group
**Dataset:** US Monthly Candy Production (IPG3113N), Kaggle / FRED

This notebook implements the complete analytical pipeline, covering exploratory analysis, stationarity testing (ADF), seasonal and first differencing, ACF/PACF analysis, model fitting across an ARIMA/SARIMA grid, model evaluation (MAE, RMSE, MAPE), residual diagnostics (Ljung-Box, Q-Q plots), and out-of-sample forecasting.

## 1. Environment Setup & Data Loading

In [ ]:
import json
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
import sarima_engine as se

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False
})

OUT = "./"
# Ensure 'candy_production.csv' is in your working directory
df = pd.read_csv(OUT + "candy_production.csv", parse_dates=["observation_date"])
dates = pd.DatetimeIndex(df["observation_date"])
y = df["IPG3113N"].values.astype(float)
s = pd.Series(y, index=dates)
R = {}
NTR = 524  # train: Jan 1972 - Aug 2015 ; test: Sep 2015 - Aug 2017 (24 months)

## 2. Descriptive Analysis & Exploratory Visualizations

In [ ]:
R["n"] = len(y); R["mean"] = float(y.mean()); R["sd"] = float(y.std())
R["min"] = float(y.min()); R["min_date"] = str(dates[y.argmin()].date())
R["max"] = float(y.max()); R["max_date"] = str(dates[y.argmax()].date())
yr = y[:540].reshape(45, 12)
R["annual_mean"] = {str(1972 + i): float(yr[i].mean()) for i in range(45)}
R["month_means"] = [float(v) for v in yr.mean(0)]
R["season_range_early"] = float((yr[:9].max(1) - yr[:9].min(1)).mean())
R["season_range_late"] = float((yr[36:45].max(1) - yr[36:45].min(1)).mean())
R["sep_oct_jump_early"] = float(np.mean([yr[i, 9] - yr[i, 8] for i in range(0, 9)]))
R["sep_oct_jump_late"] = float(np.mean([yr[i, 9] - yr[i, 8] for i in range(36, 45)]))
rm = s.rolling(12).mean(); rs = s.rolling(12).std()
R["rm_min"], R["rm_max"] = float(rm.min()), float(rm.max())
R["rs_min"], R["rs_max"] = float(rs.min()), float(rs.max())
R["cv_early"] = float(np.mean([yr[i].std() / yr[i].mean() for i in range(0, 9)]))
R["cv_late"] = float(np.mean([yr[i].std() / yr[i].mean() for i in range(36, 45)]))

# Fig 1
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(s.index[:NTR], s.values[:NTR], color="#1f77b4", lw=1, label="Training (Jan 1972 - Aug 2015)")
ax.plot(s.index[NTR - 1:], s.values[NTR - 1:], color="#d62728", lw=1.2, label="Test (Sep 2015 - Aug 2017)")
ax.set_title("US Candy Production Index (not seasonally adjusted), 1972-2017")
ax.set_ylabel("Index"); ax.set_xlabel("Year"); ax.legend(); plt.tight_layout(); plt.savefig(OUT + "f1_original.png"); plt.close()

# Fig 2: seasonal box plot by calendar month
mn = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
bym = [y[i::12] for i in range(12)]
fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot(bym, tick_labels=mn, patch_artist=True, boxprops=dict(facecolor="#9ecae1"), medianprops=dict(color="k"))
ax.set_title("Distribution of Production Index by Calendar Month (1972-2017)"); ax.set_ylabel("Index")
plt.tight_layout(); plt.savefig(OUT + "f2_seasonal.png"); plt.close()

# Fig 3: rolling mean / std
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(s.index, s.values, color="#cccccc", label="Index")
ax.plot(rm.index, rm.values, color="#1f77b4", lw=2, label="12-month rolling mean")
ax.plot(rs.index, rs.values, color="#d62728", lw=2, label="12-month rolling std")
ax.set_title("Rolling Mean and Standard Deviation (window = 12)"); ax.legend()
plt.tight_layout()
plt.show()
plt.close()

/var/folders/4p/27f0cm2j5x787xt6591nc9dh0000gn/T/ipykernel_12155/2237473056.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Stationarity Testing, Differencing, and ACF/PACF Analysis

In [ ]:
def corr_plot(x, nlags, title, color, path, pacf=False):
    v = se.pacf(x, nlags) if pacf else se.acf(x, nlags)
    ci = 1.96 / np.sqrt(len(x))
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.vlines(range(nlags + 1), 0, v, color=color); ax.plot(range(nlags + 1), v, "o", color=color, ms=3.5)
    ax.axhline(0, color="k", lw=0.8); ax.axhspan(-ci, ci, color=color, alpha=0.15)
    ax.set_title(title); ax.set_xlabel("Lag"); ax.set_ylabel("Partial autocorrelation" if pacf else "Autocorrelation")
    plt.tight_layout()
    plt.show()
    plt.close()
    return v

a0 = corr_plot(y, 36, "ACF of the Original Series", "#1f77b4", OUT + "f4_acf_original.png")
R["acf_orig"] = {str(k): float(a0[k]) for k in (1, 2, 6, 12, 24, 36)}

# ---------- differencing + ADF
fd = np.diff(y); sd = y[12:] - y[:-12]; sfd = np.diff(sd)
fig, axes = plt.subplots(3, 1, figsize=(9, 8))
axes[0].plot(dates[1:], fd, color="#1f77b4", lw=0.8); axes[0].set_title("First Difference (d = 1)")
axes[1].plot(dates[12:], sd, color="#2ca02c", lw=0.8); axes[1].set_title("Seasonal Difference (D = 1, m = 12)")
axes[2].plot(dates[13:], sfd, color="#9467bd", lw=0.8); axes[2].set_title("Seasonal + First Difference (d = 1, D = 1)")
for a in axes: a.axhline(0, color="k", lw=0.8)
plt.tight_layout(); plt.savefig(OUT + "f5_differenced.png"); plt.close()

R["adf"] = {}
for name, x in [("Original series", y), ("First difference (d=1)", fd),
                ("Seasonal difference (D=1)", sd), ("Seasonal + first difference (d=1, D=1)", sfd)]:
    a = se.adf(x); R["adf"][name] = dict(stat=float(a["stat"]), p=float(a["pvalue"]), lag=int(a["lag"])) 
R["acf_fd"] = {"1": float(se.acf(fd, 36)[1]), "12": float(se.acf(fd, 36)[12])}
R["acf_sd"] = [float(v) for v in se.acf(sd, 36)[1:6]]
R["pacf_sd"] = [float(v) for v in se.pacf(sd, 36)[1:4]]

fa = corr_plot(sfd, 36, "ACF of the Seasonally + First Differenced Series", "#1f77b4", OUT + "f6_acf.png")
fp = corr_plot(sfd, 36, "PACF of the Seasonally + First Differenced Series", "#ff7f0e", OUT + "f7_pacf.png", pacf=True)
R["acf_sfd"] = [float(v) for v in fa[1:25]]; R["pacf_sfd"] = [float(v) for v in fp[1:25]]
R["band_sfd"] = float(1.96 / np.sqrt(len(sfd)))

## 4. Model Fitting, Evaluation, Grid Search & Comparison

In [ ]:
train, test = y[:NTR], y[NTR:]
test_dates = dates[NTR:]
specs = {
    "ARIMA(1,1,1)": ((1, 1, 1), (0, 0, 0, 0)),
    "ARIMA(2,1,2)": ((2, 1, 2), (0, 0, 0, 0)),
    "SARIMA(1,0,0)(0,1,1)12": ((1, 0, 0), (0, 1, 1, 12)),
    "SARIMA(0,1,1)(0,1,1)12": ((0, 1, 1), (0, 1, 1, 12)),
    "SARIMA(1,1,1)(0,1,1)12": ((1, 1, 1), (0, 1, 1, 12)),
    "SARIMA(3,1,3)(0,1,1)12": ((3, 1, 3), (0, 1, 1, 12)),
}
BEST = "SARIMA(3,1,3)(0,1,1)12"
RUNNER = "SARIMA(1,1,1)(0,1,1)12"
fits, fc = {}, {}
R["models"] = {}
for name, (o, so) in specs.items():
    m = se.SARIMA(o, so).fit(train)
    f, lo, hi = m.forecast(24)
    fits[name] = m; fc[name] = (f, lo, hi)
    mt = se.metrics(test, f); r = m.resid
    q12, q24 = se.ljung_box(r, 12, m.k), se.ljung_box(r, 24, m.k)
    ra = se.acf(r, 36); band = 1.96 / np.sqrt(len(r))
    R["models除了name"] = name
    R["models"][name] = dict(
        params=[(n, float(b), float(sd_), float(b / sd_), float(2 * (1 - stats.norm.cdf(abs(b / sd_))))) 
                for n, b, sd_ in zip(m.param_names(), m.params, m.bse)],
        sigma=float(np.sqrt(m.sigma2)), nresid=int(len(r)),
        MAE=float(mt["MAE"]), MSE=float(mt["MSE"]), RMSE=float(mt["RMSE"]), MAPE=float(mt["MAPE"]),
        lb12=[float(q12[0]), float(q12[1]), int(q12[2])], lb24=[float(q24[0]), float(q24[1]), int(q24[2])],
        coverage=float(np.mean((test >= lo) & (test <= hi))), bias=float(np.mean(test - f)),
        forecast=[float(v) for v in f], lo=[float(v) for v in lo], hi=[float(v) for v in hi],
        shapiro=[float(stats.shapiro(r).statistic), float(stats.shapiro(r).pvalue)],
        resid_acf_out=[[int(k), float(ra[k])] for k in range(1, 37) if abs(ra[k]) > band],
        max_abs_resid_acf=float(np.max(np.abs(ra[1:25]))),
    )
sn = np.tile(train[-12:], 2); mt = se.metrics(test, sn)
R["models"]["Seasonal naive (baseline)"] = dict(MAE=float(mt["MAE"]), MSE=float(mt["MSE"]), RMSE=float(mt["RMSE"]),
                                            MAPE=float(mt["MAPE"]), forecast=[float(v) for v in sn])
R["test"] = [float(v) for v in test]; R["test_dates"] = [d.strftime("%b %Y") for d in test_dates]

# residual volatility by calendar month (recommended model)
mb = fits[BEST]
rf = mb.resid_full
mon = dates[:NTR][13:].month
R["resid_sd_by_month"] = {int(k): float(v) for k, v in pd.Series(rf).groupby(mon).std().items()}

SKIP = 13 + 6
R["ic"] = {}
for name in ("SARIMA(0,1,1)(0,1,1)12", "SARIMA(1,1,1)(0,1,1)12", "SARIMA(3,1,3)(0,1,1)12"):
    mm = fits[name]; rr = mm.resid_full[SKIP:]; n = len(rr); k = mm.k
    R["ic"][name] = dict(AIC=float(n * np.log(np.mean(rr ** 2)) + 2 * k), BIC=float(n * np.log(np.mean(rr ** 2)) + np.log(n) * k),
                         k=int(k), insample_rmse=float(np.sqrt(np.mean(rr ** 2))), n=int(n))
mm = fits[BEST]; phi = mm.params[:3]; th = mm.params[3:6]
R["best_roots"] = dict(ar=[float(v) for v in np.abs(np.roots(np.r_[-phi[::-1], 1]))],
                       ma=[float(v) for v in np.abs(np.roots(np.r_[th[::-1], 1]))])

# grid of ARMA orders around seasonal MA(1)
grid = []
for p_ in range(0, 5):
    for q_ in range(0, 5):
        if p_ == 0 and q_ == 0: continue
        g = se.SARIMA((p_, 1, q_), (0, 1, 1, 12)).fit(train)
        rr = g.resid_full[SKIP:]; n = len(rr)
        grid.append(dict(p=p_, q=q_, AIC=float(n * np.log(np.mean(rr ** 2)) + 2 * g.k),
                         BIC=float(n * np.log(np.mean(rr ** 2)) + np.log(n) * g.k),
                         lb24=float(se.ljung_box(g.resid, 24, g.k)[1]), k=int(g.k),
                         rmse=float(se.metrics(test, g.forecast(24)[0])["RMSE"])))
R["grid"] = sorted(grid, key=lambda d: d["BIC"])

## 5. Residual Diagnostics & Out-of-Sample Forecasting

In [ ]:
# Fig 8: test forecasts
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.plot(dates[NTR - 36:NTR], train[-36:], color="#888888", label="Training (last 3 years)")
ax.plot(test_dates, test, color="k", lw=2, marker="o", ms=3.5, label="Actual (test)")
ax.plot(test_dates, fc["ARIMA(1,1,1)"][0], color="#d62728", ls="--", label="ARIMA(1,1,1)")
ax.plot(test_dates, fc[RUNNER][0], color="#2ca02c", ls="-.", label=RUNNER)
ax.plot(test_dates, fc[BEST][0], color="#1f77b4", lw=2, label=BEST)
ax.fill_between(test_dates, fc[BEST][1], fc[BEST][2], color="#1f77b4", alpha=0.12, label="95% interval (recommended)")
ax.set_title("Test-Period Forecasts (Sep 2015 - Aug 2017) vs Actual"); ax.set_ylabel("Index"); ax.legend(fontsize=7.5, ncol=2)
plt.tight_layout()
plt.show()
plt.close()

# Fig 9: residual diagnostics
r = mb.resid
fig, axes = plt.subplots(2, 2, figsize=(9, 6.2))
axes[0, 0].plot(r, color="#1f77b4", lw=0.7); axes[0, 0].axhline(0, color="k", lw=0.8); axes[0, 0].set_title("Residuals over time")
ra = se.acf(r, 36); ci = 1.96 / np.sqrt(len(r))
axes[0, 1].vlines(range(1, 37), 0, ra[1:], color="#1f77b4"); axes[0, 1].plot(range(1, 37), ra[1:], "o", ms=3, color="#1f77b4")
axes[0, 1].axhspan(-ci, ci, color="#1f77b4", alpha=0.15); axes[0, 1].set_title("ACF of residuals (lag 0 omitted)")
axes[1, 0].hist(r, bins=30, color="#9ecae1", edgecolor="white", density=True)
xs = np.linspace(r.min(), r.max(), 100); axes[1, 0].plot(xs, stats.norm.pdf(xs, r.mean(), r.std()), "r")
axes[1, 0].set_title("Histogram with normal curve")
stats.probplot(r, dist="norm", plot=axes[1, 1]); axes[1, 1].set_title("Normal Q-Q plot")
plt.tight_layout()
plt.show()
plt.close()

# ---------- refit on all data, forecast the next 12 months
mf = se.SARIMA((3, 1, 3), (0, 1, 1, 12)).fit(y)
ff, flo, fhi = mf.forecast(12)
R["final_params"] = [float(v) for v in mf.params]
R["future"] = [float(v) for v in ff]; R["future_lo"] = [float(v) for v in flo]; R["future_hi"] = [float(v) for v in fhi]
fd_ = pd.date_range("2017-09-01", periods=12, freq="MS")
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(dates[-48:], y[-48:], color="k", label="Actual (Sep 2013 - Aug 2017)")
ax.plot(fd_, ff, color="#1f77b4", lw=2, marker="o", ms=4, label="Forecast Sep 2017 - Aug 2018")
ax.fill_between(fd_, flo, fhi, color="#1f77b4", alpha=0.15, label="95% interval")
ax.set_title("SARIMA(3,1,3)(0,1,1)12 Forecast (model refit on all data)"); ax.set_ylabel("Index"); ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
plt.close()

json.dump(R, open(OUT + "results.json", "w"), indent=1)
print("OK - Execution Complete!")
for k, v in R["models"].items():
    if isinstance(v, dict) and "MAE" in v:
        print(f"{k:28s} MAE={v['MAE']:.2f} RMSE={v['RMSE']:.2f} MAPE={v['MAPE']:.2f}%")